# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**What the queue answers:** *which visible pages should an editor review first, because they look likely to keep under-capturing CTR for their position, and how many clicks are at stake?*

**How it's built (frozen from Weeks 4–6, nothing new tuned here):**
- **Model:** the Week-5 random forest, same features, retrained on all clients for *feature month → next month*. It is then applied to the **latest** month to score the month ahead. That's the deployment step; no labels exist for it yet.
- **Priority = P(stays under) × missed clicks.** Missed clicks = clicks the page would get at its position bucket's pooled CTR, minus the clicks it got. The probability says *is the gap real*, the missed clicks say *is it worth an editor's time*.
- **Receipt:** before trusting it, the cell re-validates this exact ranking time-forward on unseen clients (train on the earlier month pair, score the next month), against the frozen Week-4 rule.

**Reason codes**, so a human sees *why* a page is on the list:

| Code | Meaning |
|---|---|
| `LOW_CTR_FOR_POSITION` | CTR last month < half of its position bucket's pooled CTR |
| `GAP_TWO_MONTHS` | The gap was there in the month before too |
| `TOP_POSITION` | Average position 1–5: visible, but the snippet isn't winning the click |
| `HIGH_VOLUME` | Top quarter of the queue by impressions |
| `AGING_PAGE` | Content older than 365 days |
| `CAUTION_THIN_DATA` | Zero clicks on < 1,500 impressions: the gap could be chance |
| `CAUTION_IMPRESSION_SHIFT` | Impressions more than halved or doubled vs. the month before: the query mix likely changed |
| `NAVIGATIONAL` | Navigational intent: low CTR is often expected (brand/site searches) |

**Archetype → action** (first matching row wins):

| Archetype | Action | Why |
|---|---|---|
| Navigational page | `MONITOR_ONLY` | Low CTR is often by design; a rewrite rarely helps |
| No current gap | `NO_ACTION` | CTR is at least half of expected |
| Thin data | `MONITOR_ONE_MONTH` | Zero clicks on few impressions can be chance (Week-6 false positives) |
| Impression shift | `CHECK_QUERY_MIX` | The worst Week-6 errors had big impression swings; look at the queries before editing |
| Persistent gap, top position | `REWRITE_TITLE_META` | Visible but losing the click two months running: snippet is the likely lever |
| Persistent gap, aging page | `REWRITE_TITLE_META + FRESHNESS_REVIEW` | Older pages hold gaps more often (decay insight below) |
| Persistent gap, positions 5–20 | `REWRITE_TITLE_META + INTENT_CHECK` | Lower visibility: check the page answers the query before polishing the snippet |
| New gap (this month only) | `RECHECK_NEXT_MONTH` | About half of one-month gaps don't repeat (decay insight below) |

In [1]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")
SEED = 42
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
for m in ["clicks", "impressions", "sessions"]:
    df[f"{m}_early_30d"] = df[f"{m}_90d"] - df[f"{m}_prev_30d"] - df[f"{m}_last_30d"]
BUCKETS = ([0, 3, 5, 10, 15, 20], ["1-3", "3-5", "5-10", "10-15", "15-20"])
vis = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 1000)].copy()
vis["bucket"] = pd.cut(vis["avg_position"], *BUCKETS[:1], labels=BUCKETS[1])

def add_window(d, w):
    d[f"ctr_{w}"] = 100 * d[f"clicks_{w}_30d"] / d[f"impressions_{w}_30d"]
    t = d.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    d[f"exp_{w}"] = d["bucket"].map(100 * t[f"clicks_{w}_30d"] / t[f"impressions_{w}_30d"]).astype(float)
    d[f"missed_{w}"] = (d[f"exp_{w}"] / 100 * d[f"impressions_{w}_30d"] - d[f"clicks_{w}_30d"]).clip(lower=0)
    d[f"under_{w}"] = (d[f"ctr_{w}"] < 0.5 * d[f"exp_{w}"]).astype(int)
    return d

def features(d, w):
    X = pd.DataFrame({
        "log_imp": np.log1p(d[f"impressions_{w}_30d"]), "log_clk": np.log1p(d[f"clicks_{w}_30d"]),
        "ctr": d[f"ctr_{w}"], "ratio": d[f"ctr_{w}"] / d[f"exp_{w}"], "avg_position": d["avg_position"],
        "log_sess": np.log1p(d[f"sessions_{w}_30d"]), "content_age_days": d["content_age_days"],
        "has_word_count": d["word_count"].notna().astype(int), "word_count": d["word_count"].fillna(0)}, index=d.index)
    return pd.concat([X, pd.get_dummies(d[["main_intent", "content_type"]], dummy_na=True, dtype=int)], axis=1)

make_rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
p_at = lambda y, s, k: y[np.argsort(-s, kind="mergesort")[:k]].mean()
gap_at = lambda g, s, k: g[np.argsort(-s, kind="mergesort")[:k]].sum() / g.sum()

# --- Receipt: time-forward + grouped validation of THIS ranking (priority = p x missed clicks) ---
v = vis[(vis["impressions_early_30d"] >= 100) & (vis["impressions_prev_30d"] >= 100) & (vis["impressions_last_30d"] >= 100)].copy()
for w in ["early", "prev", "last"]:
    v = add_window(v, w)
v = v.reset_index(drop=True)
Xa, Xb = features(v, "early").align(features(v, "prev"), join="outer", axis=1, fill_value=0)
y_next = v["under_last"].values
gap_next = np.where(y_next == 1, v["missed_last"], 0.0)
rows = []
for tr, te in GroupKFold(5).split(Xb, y_next, v["client_id"]):
    p = make_rf().fit(Xa.iloc[tr], v["under_prev"].iloc[tr]).predict_proba(Xb.iloc[te])[:, 1]
    rank = {"Frozen Week-4 rule": np.where(v["under_prev"].values[te] == 1, v["missed_prev"].values[te], 0.0),
            "Playbook queue (p x missed clicks)": p * v["missed_prev"].values[te]}
    for name, s in rank.items():
        rows.append({"ranking": name, "precision@50": p_at(y_next[te], s, 50),
                     "clicks_gap_captured@50": gap_at(gap_next[te], s, 50), "base_rate": y_next[te].mean()})
receipt = pd.DataFrame(rows).groupby("ranking").mean()
print("Receipt - time-forward, unseen clients, 5 folds (means):")
print(receipt.round(3).to_string())

# --- Train on all clients (prev -> last), score the latest month ---
tr_df = add_window(add_window(vis[(vis["impressions_prev_30d"] >= 100) & (vis["impressions_last_30d"] >= 100)].copy(), "prev"), "last")
score_df = vis[vis["impressions_last_30d"] >= 100].copy()
score_df = add_window(score_df, "last")
score_df["under_prev"] = np.nan
has_prev = score_df["impressions_prev_30d"] >= 100
score_df.loc[has_prev, "under_prev"] = add_window(score_df[has_prev].copy(), "prev")["under_prev"]
X_tr, X_now = features(tr_df, "prev").align(features(score_df, "last"), join="outer", axis=1, fill_value=0)
model = make_rf().fit(X_tr, tr_df["under_last"])
q = score_df.assign(p_stays_under=model.predict_proba(X_now)[:, 1])
q["priority"] = q["p_stays_under"] * q["missed_last"]
q["imp_change"] = q["impressions_last_30d"] / q["impressions_prev_30d"].replace(0, np.nan)

# --- Reason codes ---
hi_vol = q.loc[q["under_last"] == 1, "impressions_last_30d"].quantile(0.75)
codes = {
    "LOW_CTR_FOR_POSITION": q["under_last"] == 1,
    "GAP_TWO_MONTHS": (q["under_last"] == 1) & (q["under_prev"] == 1),
    "TOP_POSITION": q["avg_position"] <= 5,
    "HIGH_VOLUME": q["impressions_last_30d"] >= hi_vol,
    "AGING_PAGE": q["content_age_days"] > 365,
    "CAUTION_THIN_DATA": (q["clicks_last_30d"] == 0) & (q["impressions_last_30d"] < 1500),
    "CAUTION_IMPRESSION_SHIFT": (q["imp_change"] < 0.5) | (q["imp_change"] > 2),
    "NAVIGATIONAL": q["main_intent"] == "navigational",
}
C = pd.DataFrame(codes)
q["reason_codes"] = C.apply(lambda r: ";".join(k for k, on in r.items() if on), axis=1)

# --- Archetype -> action (first match wins) ---
persistent = C["GAP_TWO_MONTHS"]
rules = [
    ("Navigational page", "MONITOR_ONLY", C["NAVIGATIONAL"]),
    ("No current gap", "NO_ACTION", ~C["LOW_CTR_FOR_POSITION"]),
    ("Thin data", "MONITOR_ONE_MONTH", C["CAUTION_THIN_DATA"]),
    ("Impression shift", "CHECK_QUERY_MIX", C["CAUTION_IMPRESSION_SHIFT"]),
    ("Persistent gap, top position", "REWRITE_TITLE_META", persistent & C["TOP_POSITION"]),
    ("Persistent gap, aging page", "REWRITE_TITLE_META + FRESHNESS_REVIEW", persistent & C["AGING_PAGE"]),
    ("Persistent gap, positions 5-20", "REWRITE_TITLE_META + INTENT_CHECK", persistent),
    ("New gap (this month only)", "RECHECK_NEXT_MONTH", pd.Series(True, index=q.index)),
]
q["archetype"], q["action"] = None, None
for arche, action, mask in rules:
    m = mask & q["archetype"].isna()
    q.loc[m, "archetype"], q.loc[m, "action"] = arche, action
q = q.sort_values("priority", ascending=False, kind="mergesort").reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)
EDIT_ACTIONS = {"REWRITE_TITLE_META", "REWRITE_TITLE_META + FRESHNESS_REVIEW", "REWRITE_TITLE_META + INTENT_CHECK"}
print(f"\nScored pages (latest month): {len(q):,} from {q['client_id'].nunique()} clients; "
      f"with a current gap: {C['LOW_CTR_FOR_POSITION'].sum():,}")
print("\nTop 100 of the queue by archetype:")
print(q.head(100).groupby(["archetype", "action"]).size().sort_values(ascending=False).to_string())
print("\nTop 10 (no IDs):")
q.head(10)[["rank", "archetype", "action", "reason_codes", "p_stays_under", "missed_last", "bucket", "impressions_last_30d"]].round(2)

Receipt - time-forward, unseen clients, 5 folds (means):
                                    precision@50  clicks_gap_captured@50  base_rate
ranking                                                                            
Frozen Week-4 rule                         0.696                   0.386      0.355
Playbook queue (p x missed clicks)         0.712                   0.419      0.355



Scored pages (latest month): 9,752 from 28 clients; with a current gap: 3,687

Top 100 of the queue by archetype:
archetype                       action                               
Persistent gap, positions 5-20  REWRITE_TITLE_META + INTENT_CHECK        38
Persistent gap, top position    REWRITE_TITLE_META                       24
Persistent gap, aging page      REWRITE_TITLE_META + FRESHNESS_REVIEW    14
Impression shift                CHECK_QUERY_MIX                          13
New gap (this month only)       RECHECK_NEXT_MONTH                        8
No current gap                  NO_ACTION                                 3

Top 10 (no IDs):


,rank,archetype,action,reason_codes,p_stays_under,missed_last,bucket,impressions_last_30d
0,1,"Persistent gap, top position",REWRITE_TITLE_META,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;TOP_POSITI...,0.75,928.29,1-3,168958
1,2,Impression shift,CHECK_QUERY_MIX,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;TOP_POSITI...,0.70,688.01,1-3,122303
2,3,"Persistent gap, positions 5-20",REWRITE_TITLE_META + INTENT_CHECK,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;HIGH_VOLUME,0.91,287.09,5-10,106985
3,4,"Persistent gap, top position",REWRITE_TITLE_META,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;TOP_POSITI...,0.71,328.12,3-5,85885
4,5,"Persistent gap, positions 5-20",REWRITE_TITLE_META + INTENT_CHECK,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;HIGH_VOLUME,0.80,263.06,5-10,99337
5,6,"Persistent gap, top position",REWRITE_TITLE_META,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;TOP_POSITI...,0.52,359.62,3-5,120791
6,7,"Persistent gap, positions 5-20",REWRITE_TITLE_META + INTENT_CHECK,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;HIGH_VOLUME,0.89,190.65,5-10,63326
7,8,"Persistent gap, positions 5-20",REWRITE_TITLE_META + INTENT_CHECK,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;HIGH_VOLUME,0.89,174.19,5-10,74135
8,9,"Persistent gap, positions 5-20",REWRITE_TITLE_META + INTENT_CHECK,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;HIGH_VOLUME,0.90,144.89,5-10,51115
9,10,"Persistent gap, positions 5-20",REWRITE_TITLE_META + INTENT_CHECK,LOW_CTR_FOR_POSITION;GAP_TWO_MONTHS;HIGH_VOLUME,0.90,136.44,5-10,55948


**Reading the receipt:** on unseen clients, one month ahead, the playbook ranking measured about the same precision@50 as the frozen rule (about 0.71 vs 0.70) and captured a slightly larger share of the missed clicks (about 0.42 vs 0.39). Both are directional differences within fold noise. The queue's value over the rule is mainly the **reason codes and archetypes** that tell an editor *what* to do; the rank order itself is similar. Of 100 top pages, a few show `NO_ACTION`: the model still expects a gap, but this month's CTR is above half of expected, so they're watch-only.

**The decay / refresh insight** (descriptive, in this data; pages with ≥ 100 impressions in each of three consecutive months). This is what the archetype rules lean on:

In [2]:
hist = v.assign(history=np.select([(v["under_early"] == 1) & (v["under_prev"] == 1), v["under_prev"] == 1, v["under_early"] == 1],
                                 ["gap 2 months running", "gap last month only", "gap 2 months ago only"], "no gap"))
persist = hist.groupby("history")["under_last"].agg(share_under_next_month="mean", pages="size") \
    .reindex(["no gap", "gap 2 months ago only", "gap last month only", "gap 2 months running"])
print("Share of pages under-capturing next month, by gap history:")
print(persist.round(3).to_string())

v["age_band"] = pd.cut(v["content_age_days"], [0, 180, 365, 10_000], labels=["<=180 days", "181-365 days", "365+ days"])
age = v.groupby("age_band", observed=True).agg(share_with_gap=("under_last", "mean"), pages=("under_last", "size"))
age["gap_persists_if_flagged"] = v[v["under_prev"] == 1].groupby("age_band", observed=True)["under_last"].mean()
print("\nBy content age:")
print(age.round(3).to_string())

flag = v[v["under_prev"] == 1]
upd = flag.groupby(np.where(flag["days_since_last_update"] <= 30, "updated <= 30 days ago", "not updated in 30 days"))["under_last"] \
    .agg(gap_persists="mean", pages="size")
print("\nFlagged pages - persistence by recorded update (workflow flag, not a measured edit):")
print(upd.round(3).to_string())

Share of pages under-capturing next month, by gap history:
                       share_under_next_month  pages
history                                             
no gap                                  0.145   4319
gap 2 months ago only                   0.441   1206
gap last month only                     0.499   1400
gap 2 months running                    0.662   2700

By content age:
              share_with_gap  pages  gap_persists_if_flagged
age_band                                                    
<=180 days             0.347   3994                    0.577
181-365 days           0.359   3332                    0.594
365+ days              0.462   2299                    0.664

Flagged pages - persistence by recorded update (workflow flag, not a measured edit):
                        gap_persists  pages
not updated in 30 days         0.614   1586
updated <= 30 days ago         0.601   2514


**What this shows (observed, not causal):**
- **A gap that repeats is the strongest signal.** Pages under-capturing two months running stayed under next month about two-thirds of the time, vs. about half for a one-month gap and about 1 in 7 for pages with no gap. That's why a *new* gap gets `RECHECK_NEXT_MONTH` and a *persistent* one gets an edit.
- **Older pages carry more gaps, and keep them.** The share with a gap and the persistence both rise with content age. This is the decay pattern, and it's why aging pages get a freshness review on top of the snippet rewrite.
- **The recorded update doesn't separate the groups.** Flagged pages updated in the last 30 days kept their gap about as often as the rest. The update flag is a workflow event (Week 6: its values are heaped on a few numbers), and updates aren't targeted at CTR, so this is **not** evidence that refreshes don't work. It means this data can't measure a refresh effect. That needs a before/after design, which is the honest next step.

## 2. Intended use and limits

**Intended use**
- **Who:** a content editor or SEO lead at FlyRank, reviewing one client's or a small portfolio's pages once a month.
- **What for:** to **order the review** of pages that are visible (positions 1–20) but appear to under-capture clicks for their position. The output is a ranked list with reasons, a *decision-support* tool.
- **What it is not:** not a forecast of clicks after an edit, not proof that a rewrite works, not a ranking of client quality, and not a production system.

**Measured performance to quote** (time-forward, unseen clients; see the receipt in section 1 and the cost/value below): the top 50 contains real persisting gaps at roughly the rate printed, vs. a base rate of about 36–38%. Expect about 3 in 10 top-50 pages to recover without any edit.

**Where it stops being valid**
- Pages outside positions 1–20, or with < 1,000 impressions in 90 days / < 100 in the scoring month.
- New clients from a very different vertical than the 28 training clients. Fold spread in Week 6 was wide (precision@50 0.60–0.94).
- `avg_position` is a 90-day average: if a page moved a lot recently, its bucket and expected CTR are off.
- Expected CTR is pooled within this portfolio, not an industry benchmark. A query mix full of brand or SERP-feature results shifts what "normal" CTR is.
- The starter slice has no query-level data, so the model can't see the impression-mix shifts behind its worst errors.

**Cost / value** (assumptions are labeled; the recovery rate has **not** been measured):

In [3]:
MINUTES = {"REWRITE_TITLE_META": 20, "REWRITE_TITLE_META + FRESHNESS_REVIEW": 45, "REWRITE_TITLE_META + INTENT_CHECK": 40,
           "CHECK_QUERY_MIX": 15, "RECHECK_NEXT_MONTH": 3, "MONITOR_ONE_MONTH": 3, "MONITOR_ONLY": 2, "NO_ACTION": 0}   # ASSUMED editor time
TOP = 50
top = q.head(TOP).copy()
top["minutes"] = top["action"].map(MINUTES)
hours = top["minutes"].sum() / 60
at_stake = top["missed_last"].sum()
expected_real = (top["p_stays_under"] * top["missed_last"]).sum()
rng = np.random.default_rng(SEED)
rand_stake = np.mean([q[q["under_last"] == 1].sample(TOP, random_state=int(s))["missed_last"].sum() for s in rng.integers(0, 10**6, 200)])

prec = receipt.loc["Playbook queue (p x missed clicks)", "precision@50"]
print(f"Top {TOP} of the queue: {hours:.1f} editor hours (assumed minutes per action)")
print(f"  clicks/month at stake (gap to bucket CTR): {at_stake:,.0f}   vs {rand_stake:,.0f} for 50 random gap pages")
print(f"  of which in gaps the model expects to persist: {expected_real:,.0f}")
print(f"  measured precision@50 (time-forward): {prec:.0%} -> about {round((1 - prec) * TOP)} of {TOP} reviews land on pages that recover anyway")
print("\nIF an edit recovered this share of the gap (scenario, NOT measured):")
for r in (0.10, 0.25, 0.50):
    print(f"  {r:>4.0%}: ~{expected_real * r:,.0f} clicks/month, ~{expected_real * r / hours:,.0f} clicks/month per editor hour")
print("\nActions in the top 50:", top["action"].value_counts().to_dict())

Top 50 of the queue: 22.8 editor hours (assumed minutes per action)
  clicks/month at stake (gap to bucket CTR): 7,808   vs 416 for 50 random gap pages
  of which in gaps the model expects to persist: 5,672
  measured precision@50 (time-forward): 71% -> about 14 of 50 reviews land on pages that recover anyway

IF an edit recovered this share of the gap (scenario, NOT measured):
   10%: ~567 clicks/month, ~25 clicks/month per editor hour
   25%: ~1,418 clicks/month, ~62 clicks/month per editor hour
   50%: ~2,836 clicks/month, ~125 clicks/month per editor hour

Actions in the top 50: {'REWRITE_TITLE_META + INTENT_CHECK': 18, 'REWRITE_TITLE_META': 15, 'CHECK_QUERY_MIX': 7, 'RECHECK_NEXT_MONTH': 5, 'REWRITE_TITLE_META + FRESHNESS_REVIEW': 5}


**Reading the cost/value:** the queue concentrates far more clicks at stake into 50 reviews than 50 random gap pages (about 7.8K vs 0.4K per month). That's partly by construction, since the priority includes missed clicks; the point is that editor hours go to large pages first. The *return* on those hours depends on a recovery rate nobody has measured yet. The scenarios show the range, and the first real month of edits (with a held-back comparison group, see section 4) is what turns them into a number. Until then the honest claim is: *this list puts the biggest likely-real gaps in front of an editor first.*

## 3. Human review + the no-go list

**Before acting on any page, a person checks:**
1. **The actual queries** in Search Console for that page. Does the page match what people search? (This is what the model can't see.)
2. **The live SERP** for the top query: is CTR low because of a featured snippet, ads, a map pack or AI answer box? If so → `MONITOR_ONLY`; a title won't fix it.
3. **Reason codes with `CAUTION_*`:** confirm the gap isn't chance (thin data) or a query-mix change (impression shift) before editing.
4. **Recent changes:** was the page edited, moved, or redirected in the last 60 days? If so, wait a month.
5. **Brand and legal constraints:** the client's tone, claims they may or may not make, regulated wording.
6. **Write the change down** (date, page, what changed, reason code) so next month's check can see it.

**The no-go list: never automate these**
- ❌ **Auto-publishing title / meta rewrites** (AI-written or otherwise) from this queue. Every edit is reviewed by a person.
- ❌ **Deleting, noindexing, merging or redirecting pages** based on the score.
- ❌ **Rewriting body content automatically** for `AGING_PAGE` / `FRESHNESS_REVIEW` pages.
- ❌ **Judging clients or editors** by their pages' scores, or putting scores in client-facing reports as "performance grades".
- ❌ **Promising click gains** to a client from this list. The recovery rate isn't measured.
- ❌ **Acting on `NAVIGATIONAL`, thin-data or impression-shift pages** without the checks above.
- ❌ **Running it on pages outside the valid range** (section 2) or on a client with no history.

**Fine to automate:** rebuilding the queue monthly, computing reason codes, and the monitoring checks in section 4.

In [4]:
REVIEW_CHECKLIST = ["queries match page intent (Search Console)", "live SERP checked for snippet/ads/AI box",
                    "CAUTION codes resolved", "no edit/move/redirect in last 60 days", "brand/legal wording ok", "change logged"]
NO_GO = {"MONITOR_ONLY", "NO_ACTION", "MONITOR_ONE_MONTH", "RECHECK_NEXT_MONTH"}
q["needs_human_review"] = True                                    # every row - nothing is auto-applied
q["edit_candidate"] = q["action"].isin(EDIT_ACTIONS) & ~q["reason_codes"].str.contains("CAUTION_")
q["review_checklist"] = " | ".join(REVIEW_CHECKLIST)

t100 = q.head(100)
print(f"Top 100: {t100['edit_candidate'].sum()} edit candidates (still human-reviewed), "
      f"{t100['action'].isin(NO_GO).sum()} watch-only, {t100['action'].eq('CHECK_QUERY_MIX').sum()} need a query check first")
print("Caution codes in the top 100:")
for code in ["CAUTION_THIN_DATA", "CAUTION_IMPRESSION_SHIFT", "NAVIGATIONAL"]:
    print(f"  {code}: {t100['reason_codes'].str.contains(code).sum()}")
assert q["needs_human_review"].all()

Top 100: 76 edit candidates (still human-reviewed), 11 watch-only, 13 need a query check first
Caution codes in the top 100:
  CAUTION_THIN_DATA: 0
  CAUTION_IMPRESSION_SHIFT: 14
  NAVIGATIONAL: 0


## 4. Monitoring / retrain triggers

The queue goes stale when the world it learned from moves. Each month, before using the new queue, check:

| Check | Trigger | Response |
|---|---|---|
| **Realized precision@50** of last month's queue (pages *not* edited) | < 0.60 (below the worst Week-6 fold) or below the rule | Fall back to the Week-4 rule; investigate |
| **Label base rate** (share of scored pages under-capturing) | moves > 10 points from training | Retrain; re-check thresholds |
| **Bucket expected CTR** | any bucket moves > 25% relative month to month | Check for a SERP/layout change; retrain |
| **Feature drift (PSI)** on CTR ratio and impressions | PSI > 0.2 | Retrain |
| **New-client share** in the queue | > 30% of top 50 from clients not in training | Flag those rows lower-confidence |
| **Schema / column change** in the export | any | Stop; fix the pipeline before scoring |
| **Calendar** | every quarter | Retrain on the latest three months, re-run the Week-6 time-forward check |

**Measuring whether edits help** (the missing evidence): each month, hold back a random ~20% of edit candidates (no edit), then compare next-month CTR change for edited vs. held-back pages. That's the first design in this project that could support a "rewrites recover clicks" claim. The cell below runs the checks that can be run today (training month vs. scoring month).

In [5]:
def psi(a, b, bins=10):
    edges = np.unique(np.quantile(a, np.linspace(0, 1, bins + 1)))
    pa = np.histogram(np.clip(a, edges[0], edges[-1]), edges)[0] / len(a) + 1e-6
    pb = np.histogram(np.clip(b, edges[0], edges[-1]), edges)[0] / len(b) + 1e-6
    return float(np.sum((pa - pb) * np.log(pa / pb)))

base_train, base_now = tr_df["under_prev"].mean(), q["under_last"].mean()
exp_shift = (tr_df.groupby("bucket", observed=True)["exp_last"].first() / tr_df.groupby("bucket", observed=True)["exp_prev"].first() - 1)
checks = [
    ("Label base rate: train month vs scoring month", f"{base_train:.1%} -> {base_now:.1%}", abs(base_now - base_train) > 0.10),
    ("Max bucket expected-CTR change", f"{exp_shift.abs().max():.1%}", exp_shift.abs().max() > 0.25),
    ("PSI ctr ratio (train vs scoring)", f"{psi(X_tr['ratio'].values, X_now['ratio'].values):.3f}", psi(X_tr['ratio'].values, X_now['ratio'].values) > 0.2),
    ("PSI log impressions", f"{psi(X_tr['log_imp'].values, X_now['log_imp'].values):.3f}", psi(X_tr['log_imp'].values, X_now['log_imp'].values) > 0.2),
    ("Receipt precision@50 (time-forward)", f"{prec:.2f}", prec < 0.60),
]
mon = pd.DataFrame(checks, columns=["check", "value", "triggered"])
mon["status"] = np.where(mon["triggered"], "RETRAIN / INVESTIGATE", "ok")
print("Bucket expected-CTR change, prev -> last month:", exp_shift.round(3).to_dict())
mon

Bucket expected-CTR change, prev -> last month: {'1-3': 0.015, '3-5': 0.045, '5-10': 0.094, '10-15': 0.171, '15-20': 0.197}


,check,value,triggered,status
0,Label base rate: train month vs scoring month,42.5% -> 37.8%,False,ok
1,Max bucket expected-CTR change,19.7%,False,ok
2,PSI ctr ratio (train vs scoring),0.025,False,ok
3,PSI log impressions,0.149,False,ok
4,Receipt precision@50 (time-forward),0.71,False,ok


## 5. Exports for the paper

- `work/outputs/action_queue.csv`: the full ranked queue. **Not committed** (the leak guard blocks data files); this notebook regenerates it. It carries pseudonymous `content_id` for joining only; the paper must not print IDs.
- `work/outputs/playbook_metrics.json`: committed receipts for every number the paper will quote.
- `work/figures/*.png`: committed figures for the paper.

In [6]:
out, figs = root / "work/outputs", root / "work/figures"
out.mkdir(parents=True, exist_ok=True); figs.mkdir(parents=True, exist_ok=True)
COLS = ["rank", "content_id", "archetype", "action", "reason_codes", "priority", "p_stays_under", "missed_last",
        "bucket", "avg_position", "impressions_last_30d", "clicks_last_30d", "ctr_last", "exp_last", "main_intent",
        "content_type", "content_age_days", "edit_candidate", "needs_human_review", "review_checklist"]
queue = q[COLS].rename(columns={"missed_last": "missed_clicks_30d", "ctr_last": "ctr_30d_pct", "exp_last": "expected_ctr_30d_pct",
                                "bucket": "position_bucket"})
queue.round(4).to_csv(out / "action_queue.csv", index=False)

# Figure 1: receipt
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a, metric in zip(ax, ["precision@50", "clicks_gap_captured@50"]):
    vals = receipt[metric].reindex(["Frozen Week-4 rule", "Playbook queue (p x missed clicks)"])
    a.bar(["Week-4 rule", "Playbook queue"], vals, color=["#9aa5b1", "#2f6db5"])
    if metric == "precision@50":
        a.axhline(receipt["base_rate"].iloc[0], ls="--", c="k", lw=1, label="base rate"); a.legend(frameon=False)
    a.set_title(metric); a.set_ylim(0, 1)
    for i, x in enumerate(vals): a.text(i, x + 0.02, f"{x:.2f}", ha="center")
fig.suptitle("Time-forward validation on unseen clients (5 folds, means)", fontsize=10); fig.tight_layout()
fig.savefig(figs / "w07_queue_validation.png", dpi=150); plt.close(fig)

# Figure 2: decay / persistence
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
ax[0].barh(persist.index, persist["share_under_next_month"], color="#2f6db5")
ax[0].set_xlabel("share under-capturing next month"); ax[0].set_xlim(0, 1); ax[0].set_title("Gap history")
ax[1].bar(age.index.astype(str), age["gap_persists_if_flagged"], color="#c8793a")
ax[1].set_ylim(0, 1); ax[1].set_title("Gap persists (flagged pages), by age")
fig.suptitle("Observed in the starter slice; descriptive, not causal", fontsize=10); fig.tight_layout()
fig.savefig(figs / "w07_gap_persistence.png", dpi=150); plt.close(fig)

# Figure 3: archetypes in the top 100
arch = q.head(100)["action"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, 3.4)); ax.barh(arch.index, arch.values, color="#2f6db5")
ax.set_title("Actions in the top 100 of the queue"); ax.set_xlabel("pages"); fig.tight_layout()
fig.savefig(figs / "w07_top100_actions.png", dpi=150); plt.close(fig)

metrics = {
    "scored_pages": int(len(q)), "clients": int(q["client_id"].nunique()), "pages_with_current_gap": int(C["LOW_CTR_FOR_POSITION"].sum()),
    "receipt_time_forward_grouped": receipt.round(4).reset_index().to_dict(orient="records"),
    "persistence_by_history": persist.round(4).reset_index().to_dict(orient="records"),
    "persistence_by_age": age.round(4).reset_index().astype({"age_band": str}).to_dict(orient="records"),
    "top50": {"editor_hours_assumed": round(hours, 1), "clicks_at_stake_30d": round(float(at_stake)),
              "random_50_gap_pages_clicks_at_stake": round(float(rand_stake)), "expected_persisting_clicks_30d": round(float(expected_real))},
    "monitoring": mon[["check", "value", "status"]].to_dict(orient="records"),
    "assumptions": {"minutes_per_action": MINUTES, "recovery_rate": "NOT measured; scenarios 10/25/50%"},
}
(out / "playbook_metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
for f in ["work/outputs/action_queue.csv", "work/outputs/playbook_metrics.json", "work/figures/w07_queue_validation.png",
          "work/figures/w07_gap_persistence.png", "work/figures/w07_top100_actions.png"]:
    print(f"{f:<45} {(root / f).stat().st_size / 1024:8.1f} KB")
print(f"Queue rows: {len(queue):,}")

work/outputs/action_queue.csv                   3545.6 KB
work/outputs/playbook_metrics.json                 2.4 KB
work/figures/w07_queue_validation.png             58.7 KB
work/figures/w07_gap_persistence.png              69.8 KB
work/figures/w07_top100_actions.png               48.0 KB
Queue rows: 9,752


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.